In [1]:
images_path = '/content/drive/MyDrive/Research_Work/Dr Indu Lathwal/Dataset'

In [2]:
import os

image_paths = []
labels = []

# Define common image extensions
IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.gif', '.bmp', '.tiff', '.webp')

print(f"Scanning directory: {images_path}")

# Recursively traverse the images_path directory
for root, dirs, files in os.walk(images_path):
    # The subfolder name will serve as the class label
    # We only care about the immediate subfolder name, not the full path segment
    if root == images_path: # Skip the root directory itself, as it's not a label
        current_label = None # No label for the root directory
    else:
        current_label = os.path.basename(root)

    for file in files:
        if file.lower().endswith(IMAGE_EXTENSIONS):
            full_path = os.path.join(root, file)
            image_paths.append(full_path)
            if current_label:
                labels.append(current_label)
            else:
                # Handle cases where images might be directly in the root images_path
                # Or assign a generic label if no specific subfolder label is appropriate
                labels.append("unknown_label") # Or any suitable default for root-level images

print(f"Found {len(image_paths)} images and {len(labels)} labels.")
print("First 5 image paths:")
for i in range(min(5, len(image_paths))):
    print(image_paths[i])

print("\nFirst 5 labels:")
for i in range(min(5, len(labels))):
    print(labels[i])

Scanning directory: /content/drive/MyDrive/Research_Work/Dr Indu Lathwal/Dataset
Found 2767 images and 2767 labels.
First 5 image paths:
/content/drive/MyDrive/Research_Work/Dr Indu Lathwal/Dataset/Hariana Front/IMG_20230417_091628846_HDR.jpg
/content/drive/MyDrive/Research_Work/Dr Indu Lathwal/Dataset/Hariana Front/IMG_20230417_091627505_HDR.jpg
/content/drive/MyDrive/Research_Work/Dr Indu Lathwal/Dataset/Hariana Front/IMG_20230417_091629664_HDR.jpg
/content/drive/MyDrive/Research_Work/Dr Indu Lathwal/Dataset/Hariana Front/IMG_20230417_091630566_HDR.jpg
/content/drive/MyDrive/Research_Work/Dr Indu Lathwal/Dataset/Hariana Front/IMG_20230417_091634446_HDR.jpg

First 5 labels:
Hariana Front
Hariana Front
Hariana Front
Hariana Front
Hariana Front


In [3]:
import numpy as np
from PIL import Image

def preprocess_image(image_path, target_size=(128, 128)):
    """
    Loads an image, resizes it, converts it to a NumPy array, and normalizes pixel values.

    Args:
        image_path (str): The path to the image file.
        target_size (tuple): The desired size (width, height) for the image.

    Returns:
        numpy.ndarray: The preprocessed image as a NumPy array with pixel values normalized to [0, 1].
    """
    try:
        # Open the image
        img = Image.open(image_path)

        # Resize the image
        img = img.resize(target_size)

        # Convert the image to a NumPy array
        img_array = np.array(img)

        # Normalize pixel values to [0, 1] if it's an RGB image (3 channels)
        # If it's a grayscale image, it might have 2 dimensions, so check for that.
        if img_array.ndim == 3 and img_array.shape[2] in [3, 4]: # RGB or RGBA
            img_array = img_array / 255.0
        elif img_array.ndim == 2: # Grayscale
            img_array = img_array / 255.0
        else:
            # Handle other cases or raise an error if expected image format is different
            print(f"Warning: Unexpected image array shape for {image_path}: {img_array.shape}. Normalization might be incorrect.")

        return img_array
    except Exception as e:
        print(f"Error processing image {image_path}: {e}")
        return None

print("Image preprocessing function `preprocess_image` defined.")

Image preprocessing function `preprocess_image` defined.


In [4]:
processed_images = []
processed_labels = []

# Define the target size for the images
TARGET_SIZE = (128, 128) # e.g., 128x128 pixels

print(f"Starting image preprocessing for {len(image_paths)} images...")

for i, (image_path, label) in enumerate(zip(image_paths, labels)):
    if i % 500 == 0: # Print progress every 500 images
        print(f"Processing image {i}/{len(image_paths)}")

    preprocessed_img = preprocess_image(image_path, target_size=TARGET_SIZE)

    if preprocessed_img is not None:
        processed_images.append(preprocessed_img)
        processed_labels.append(label)

# Convert lists to NumPy arrays for efficient storage and manipulation
X = np.array(processed_images)
y = np.array(processed_labels)

print(f"Finished preprocessing. Created dataset with shape X: {X.shape}, y: {y.shape}")
print("First preprocessed image shape:", X[0].shape)
print("First label:", y[0])

Starting image preprocessing for 2767 images...
Processing image 0/2767
Processing image 500/2767
Processing image 1000/2767
Processing image 1500/2767
Processing image 2000/2767
Processing image 2500/2767
Finished preprocessing. Created dataset with shape X: (2767, 128, 128, 3), y: (2767,)
First preprocessed image shape: (128, 128, 3)
First label: Hariana Front


In [5]:
import os

image_dataset_path = '/content/drive/MyDrive/Research_Work/Dr Indu Lathwal/Dataset/'
output_filename = 'preprocessed_image_dataset.npz'

# Ensure the directory exists (optional, as it's typically an existing Drive folder)
os.makedirs(image_dataset_path, exist_ok=True)

full_output_path = os.path.join(image_dataset_path, output_filename)

# Save the processed images (X) and labels (y) to a compressed .npz file
np.savez_compressed(full_output_path, images=X, labels=y)

print(f"Dataset successfully saved to: {full_output_path}")

# You can verify by loading it back (optional)
# loaded_data = np.load(full_output_path)
# loaded_images = loaded_data['images']
# loaded_labels = loaded_data['labels']
# print(f"Verified: Loaded images shape: {loaded_images.shape}, Loaded labels shape: {loaded_labels.shape}")

Dataset successfully saved to: /content/drive/MyDrive/Research_Work/Dr Indu Lathwal/Dataset/preprocessed_image_dataset.npz
